## paths and setup

In [1]:
# % load_ext autoreload
# % autoreload 2

from pathlib import Path
import yaml
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

PROJECT_ROOT = Path.cwd().parent          # notebooks/ -> circmtl/
with open(PROJECT_ROOT / "configs" / "paths.yaml") as f:
    cfg = yaml.safe_load(f)

RAW_DIR = PROJECT_ROOT / cfg["raw_dir"]
PROCESSED_DIR = PROJECT_ROOT / cfg["processed_dir"]

print("raw dir:", RAW_DIR, "| exists:", RAW_DIR.exists())
for p in sorted(RAW_DIR.iterdir()):
    print(f"{p.stat().st_size / 1e6:8.2f} MB   {p.name}")

raw dir: d:\Queens_university\Project\Working\Notebook\circmtl\data\raw | exists: True
    2.91 MB   1-s2.0-S1672022921002461-main.pdf
    0.02 MB   encori
  500.37 MB   human_hg19_circRNAs_putative_spliced_sequence.fa
 1194.42 MB   RNA-FM_pretrained.pth
    2.12 MB   The_candidate_circRNA-disease_associations_that_predicted_by_CircDis_method.xlsx
    0.73 MB   The_circRNA-disease_entries.xlsx
    0.06 MB   The_circRNA_list.xlsx
    0.03 MB   The_disease_list.xlsx
    0.12 MB   The_manually_curated_associations_between_circRNAs_and_other_associations.xlsx


## the shape of every Excel file

In [2]:
tables = {}   # tables[file_stem][sheet_name] = DataFrame

for f in sorted(RAW_DIR.glob("*.xlsx")):
    sheets = pd.read_excel(f, sheet_name=None)   # None -> all sheets
    tables[f.stem] = sheets
    print("=" * 100)
    print(f.name)
    for sheet, d in sheets.items():
        print(f"  sheet '{sheet}': {d.shape[0]} rows x {d.shape[1]} cols")
        print(f"  columns: {list(d.columns)}")

The_candidate_circRNA-disease_associations_that_predicted_by_CircDis_method.xlsx
  sheet 'Sheet3': 1577 rows x 203 cols
  columns: ['Unnamed: 0', 'Cardiovascular disease', 'Lung cancer', 'Osteoarthritis', 'Intervertebral disc degeneration', 'Rheumatoid arthritis', 'Atherosclerosis', 'Prostate cancer', 'Glioblastoma', 'Hepatocellular carcinoma', 'Repeated implantation failure', 'Bladder cancer', 'Gastric cancer', 'Breast cancer', 'Liver cancer', 'Osteosarcoma', 'Colorectal cancer', 'Moyamoya disease', 'PulmoNAry tuberculosis', 'Triple negative breast cancer', 'Glioma', 'Cervical cancer', 'Hepatic fibrosis', 'Multiple sclerosis', 'Oral squamous cell carcinoma', 'Hypopharyngeal squamous cell carcinoma', 'Lung adenocarcinoma', 'Clear cell reNAl cell carcinoma', 'Hypertension', 'Senescence', 'Cervical carcinoma', 'Acute myeloid leukemia', 'Papillary thyroid carcinoma', 'Type 2 diabetes mellitus', 'CoroNAry artery disease', 'Hirschsprung disease', 'Pancreatic ductal adenocarcinoma', 'Preecla

## the list files and confirm the counts

In [3]:
circ_list = pd.read_excel(RAW_DIR / "The_circRNA_list.xlsx", header=None, names=["circRNA_id"])
dis_list  = pd.read_excel(RAW_DIR / "The_disease_list.xlsx",  header=None, names=["disease"])

print("circRNAs:", len(circ_list), "| unique:", circ_list["circRNA_id"].nunique())
print("diseases:", len(dis_list),  "| unique:", dis_list["disease"].nunique())

print("\nFirst circRNA IDs:")
print(circ_list["circRNA_id"].head(8).tolist())

print("\nID format check (share starting with a digit/chr vs other):")
print(circ_list["circRNA_id"].str.extract(r"^([A-Za-z]+)")[0].value_counts(dropna=False).head(10))

circRNAs: 3077 | unique: 3077
diseases: 312 | unique: 312

First circRNA IDs:
['11_33286412_33287511_+', '17_50189167_50194626_-', '17_50189167_50198002_-', '17_50189858_50195330_-', '17_50190326_50194041_-', '2_72718102_72733118_-', 'ASB3-has_circ_0003959', 'BCRC-3']

ID format check (share starting with a digit/chr vs other):
0
hsa        1825
circ        341
circRNA     213
mmu          95
Circ         79
rno          36
chr          27
CircRNA      14
NaN           6
CBT           6
Name: count, dtype: int64


## Overview of the main entries table

In [4]:
entries = tables["The_circRNA-disease_entries"]["The CircRNA-Disease Entries"]

print(entries.shape)
print("\nFirst record (transposed):")
print(entries.head(1).T)

print("\nMissing values per column:")
print(entries.isna().sum())

print("\nSpecies:")
print(entries["Species"].value_counts(dropna=False))

print("\nUnique circRNA names:", entries["circRNA Name"].nunique())
print("Unique disease names:", entries["Disease Name"].nunique())

(4201, 13)

First record (transposed):
                                                                                   0
CRD ID                                                                             1
circRNA Name                                                      hsa_circRNA_001937
Synonyms                                                                         NaN
Gene Symbol                                                                     CHD9
Disease Name                                                            Tuberculosis
Expression pattern                                                       Upregulated
PubMed ID                                                                 29448254.0
Region                                                                           NaN
Strand                                                                           NaN
Species                                                                        Human
Experimental techniques   

## Peek at the FASTA headers

In [5]:
fa = next(RAW_DIR.glob("*.fa"))
print(fa.name, f"{fa.stat().st_size / 1e6:.1f} MB")

with open(fa) as f:
    for _ in range(6):
        print(f.readline().rstrip()[:150])

human_hg19_circRNAs_putative_spliced_sequence.fa 500.4 MB
>hsa_circ_0000001|chr1:1080738-1080845-|None|None
ATGGGGTTGGGTCAGCCGTGCGGTCAGGTCAGGTCGGCCATGAGGTCAGGTGGGGTCGGCCATGAAGGTGGTGGGGGTCATGAGGTCACAAGGGGGTCGGCCATGTG
>hsa_circ_0000002|chr1:1158623-1159348-|NM_016176|SDF4
GGTGGATGTGAACACTGACCGGAAGATCAGTGCCAAGGAGATGCAGCGCTGGATCATGGAGAAGACGGCCGAGCACTTCCAGGAGGCCATGGAGGAGAGCAAGACACACTTCCGCGCCGTGGACCCTGACGGGGACGGTCACGTGTCTTG
>hsa_circ_0000003|chr1:1423242-1459777+|NM_031921|ATAD3B
CCTCCTGCTCTTCATGGATGAAGCAGACGCCTTCCTTCGGAAGCGAGCCACTGAGGAGATAAGCAAGGACCTCAGAGCCACACTGAACGCCTTCCTGTACCACATGGGCCAACACAGCAACAAATTCATGCTGGTCCTGGCCAGCAATCT


## Index the FASTA headers

In [6]:
import re

fa_path = next(RAW_DIR.glob("*.fa"))

def parse_header(h):
    # e.g. >hsa_circ_0000001|chr1:1080738-1080845-|None|None
    parts = h[1:].split("|")
    loc = parts[1] if len(parts) > 1 else ""
    m = re.match(r"^(chr\w+):(\d+)-(\d+)([+-])$", loc)
    chrom, start, end, strand = m.groups() if m else (None, None, None, None)
    return {
        "circbase_id": parts[0],
        "chrom": chrom, "start": start, "end": end, "strand": strand,
        "transcript": parts[2] if len(parts) > 2 else None,
        "gene": parts[3] if len(parts) > 3 else None,
    }

records, cur, seq_len = [], None, 0
with open(fa_path) as f:
    for line in f:
        if line.startswith(">"):
            if cur is not None:
                cur["seq_len"] = seq_len
                records.append(cur)
            cur, seq_len = parse_header(line.strip()), 0
        else:
            seq_len += len(line.strip())
    if cur is not None:
        cur["seq_len"] = seq_len
        records.append(cur)

fa_index = pd.DataFrame(records)
for c in ["start", "end"]:
    fa_index[c] = pd.to_numeric(fa_index[c], errors="coerce").astype("Int64")

fa_index.to_csv(PROCESSED_DIR / "circbase_hg19_index.csv", index=False)

print("records:", len(fa_index), "| unique IDs:", fa_index["circbase_id"].nunique())
print("header parse failures:", fa_index["chrom"].isna().sum())
print("\nstrand:\n", fa_index["strand"].value_counts(dropna=False))
print("\ngene field is 'None':", (fa_index["gene"] == "None").mean().round(3))
print("\nsequence length (nt):")
print(fa_index["seq_len"].describe())
print("share <= 1022 nt (RNA-FM limit):", (fa_index["seq_len"] <= 1022).mean().round(3))

records: 140790 | unique IDs: 140790
header parse failures: 0

strand:
 strand
-    70436
+    70354
Name: count, dtype: int64

gene field is 'None': 0.017

sequence length (nt):
count    1.407900e+05
mean     3.491749e+03
std      3.444732e+04
min      0.000000e+00
25%      3.290000e+02
50%      6.840000e+02
75%      1.545000e+03
max      1.847059e+06
Name: seq_len, dtype: float64
share <= 1022 nt (RNA-FM limit): 0.635


## Species, ID patterns, and name hygiene in the entries

In [7]:
# 1. normalise species labels
species_map = {"human": "Human", "mouse": "Mouse", "rat": "Rat", "rats": "Rat",
               "chicken": "Chicken", "c. elegans": "C. elegans"}
entries["species_clean"] = entries["Species"].str.strip().str.lower().map(species_map)
print(entries["species_clean"].value_counts(dropna=False), "\n")

# 2. classify the naming scheme of each circRNA name
def name_pattern(name):
    n = str(name).strip()
    if re.search(r"hsa_circ_\d+", n, re.I):          return "circbase_full"
    if re.match(r"^(chr)?[\dXYM]+[_:]\d+[_-]\d+", n): return "coordinate"
    if re.search(r"circ_\d+", n, re.I):              return "circbase_no_prefix"
    if re.search(r"circRNA_\d+", n, re.I):           return "arraystar_like"
    return "other"

entries["name_pattern"] = entries["circRNA Name"].map(name_pattern)
print(pd.crosstab(entries["name_pattern"], entries["species_clean"], margins=True), "\n")

# 3. what do Region / Strand look like?
print(entries.loc[entries["Region"].notna(),
                  ["circRNA Name", "Region", "Strand", "species_clean"]]
      .sample(8, random_state=0), "\n")

# 4. name hygiene: entries vs the official lists
names = entries["circRNA Name"].astype(str)
print("unique circRNA names raw:", names.nunique(),
      "| stripped+casefold:", names.str.strip().str.casefold().nunique())
d = entries["Disease Name"].astype(str)
print("unique diseases raw:", d.nunique(),
      "| stripped+casefold:", d.str.strip().str.casefold().nunique())
print("\nin entries but not in disease_list:", sorted(set(entries["Disease Name"]) - set(dis_list["disease"])))
print("in disease_list but not in entries:", sorted(set(dis_list["disease"]) - set(entries["Disease Name"])))
in_list = entries["circRNA Name"].isin(circ_list["circRNA_id"])
print("\nentry rows whose circRNA name is in circRNA_list:", in_list.sum(), "/", len(entries))

species_clean
Human         3854
Mouse          248
Rat             93
Chicken          4
C. elegans       2
Name: count, dtype: int64 

species_clean       C. elegans  Chicken  Human  Mouse  Rat   All
name_pattern                                                    
arraystar_like               0        0    339     76   38   453
circbase_full                0        0   2317      3    2  2322
circbase_no_prefix           2        0    217     61    6   286
coordinate                   0        4     14      0    0    18
other                        0        0    967    108   47  1122
All                          2        4   3854    248   93  4201 

            circRNA Name                    Region Strand species_clean
789     hsa_circ_0006215    chr3:27478878-27490288      -         Human
2811    hsa_circ_0006772      chr7:5410005-5410995      -         Human
1681    hsa_circ_0079593    chr7:23381682-23383472      -         Human
2994    hsa_circ_0006156  chr3:171965322-171969331   

## Human subset, name cleaning, and circBase-ID resolution (tier 1)

In [8]:
# ---- 1. human subset + text cleaning ------------------------------------
def clean_txt(s):
    return re.sub(r"\s+", " ", str(s)).strip()

hum = entries[entries["species_clean"] == "Human"].copy()
hum["circ_raw"]      = hum["circRNA Name"].map(clean_txt)
hum["disease_clean"] = hum["Disease Name"].map(clean_txt)

dis_std = {clean_txt(d).casefold(): clean_txt(d) for d in dis_list["disease"]}
hum["disease_std"] = hum["disease_clean"].str.casefold().map(dis_std)
print("human rows:", len(hum))
print("diseases not matched to official list:", hum["disease_std"].isna().sum())

# is the 'NA' corruption (e.g. 'PulmoNAry') also in the official list?
bad = dis_list[dis_list["disease"].str.contains(r"[a-z]NA|\bNA[a-z]", regex=True)]
print("official disease names with 'NA' corruption:", len(bad), bad["disease"].head(8).tolist())

# ---- 2. tier 1: circBase ID written in the name or in Synonyms ----------
pat = re.compile(r"(?:hsa|has)_circ_(\d{7})(?!\d)", re.I)

def ids_in(text):
    return sorted({f"hsa_circ_{m}" for m in pat.findall(text)}) if isinstance(text, str) else []

hum["ids_name"] = hum["circ_raw"].map(ids_in)
hum["ids_syn"]  = hum["Synonyms"].map(ids_in)
has_name = hum["ids_name"].map(len) > 0
has_syn  = hum["ids_syn"].map(len) > 0

print("\nrows with circBase ID in name:      ", has_name.sum())
print("rows with ID only in Synonyms:      ", (~has_name & has_syn).sum())
print("rows with >1 distinct ID (name+syn):",
      ((hum["ids_name"] + hum["ids_syn"]).map(lambda x: len(set(x))) > 1).sum())

near = hum["circ_raw"].str.contains(r"(?:hsa|has)_circ_\d+", case=False, regex=True) & ~has_name
print("rows with 'hsa_circ_' but not a clean 7-digit ID:", near.sum(),
      hum.loc[near, "circ_raw"].head(6).tolist())

def pick(row):
    cand = row["ids_name"] or row["ids_syn"]
    return cand[0] if len(cand) == 1 else None   # ambiguous -> unresolved for now

hum["circbase_id"] = hum.apply(pick, axis=1)
fa_ids = set(fa_index["circbase_id"])
hum["in_fasta"] = hum["circbase_id"].isin(fa_ids)

# ---- 3. what survives ---------------------------------------------------
res   = hum[hum["in_fasta"]]
pairs = res[["circbase_id", "disease_std"]].drop_duplicates()
print("\nID found but NOT in FASTA:", (hum["circbase_id"].notna() & ~hum["in_fasta"]).sum())
print("resolved human rows:", len(res), "/", len(hum))
print("unique circRNA-disease pairs:", len(pairs))
print("unique circRNAs:", pairs["circbase_id"].nunique(),
      "| unique diseases:", pairs["disease_std"].nunique())
print("paper's circDis universe:  2099 pairs | 1577 circRNAs | 202 diseases")

# ---- 4. FASTA quirks ----------------------------------------------------
print("\nFASTA zero-length sequences:", (fa_index["seq_len"] == 0).sum())
print("FASTA > 10,000 nt:", (fa_index["seq_len"] > 10_000).sum(),
      "| > 100,000 nt:", (fa_index["seq_len"] > 100_000).sum())

human rows: 3854
diseases not matched to official list: 0
official disease names with 'NA' corruption: 0 []

rows with circBase ID in name:       2259
rows with ID only in Synonyms:       158
rows with >1 distinct ID (name+syn): 0
rows with 'hsa_circ_' but not a clean 7-digit ID: 65 ['hsa_circ_402458', 'hsa_circ_406329', 'hsa_circ_407176', 'hsa_circ_082319', 'hsa_circ_10072088', 'hsa_circ_101396']

ID found but NOT in FASTA: 5
resolved human rows: 2412 / 3854
unique circRNA-disease pairs: 2096
unique circRNAs: 1579 | unique diseases: 200
paper's circDis universe:  2099 pairs | 1577 circRNAs | 202 diseases

FASTA zero-length sequences: 58
FASTA > 10,000 nt: 5486 | > 100,000 nt: 335


## Who is unresolved, and why

In [9]:
# A. the 5 IDs that are not in the FASTA
miss = hum[hum["circbase_id"].notna() & ~hum["in_fasta"]]
print("IDs not in FASTA:")
print(miss[["circ_raw", "circbase_id", "Region", "Strand", "Gene Symbol"]], "\n")

# B. the near-miss IDs
nm = hum.loc[near, ["circ_raw", "Synonyms", "Region", "Strand", "Gene Symbol"]]
print("near-miss rows:", len(nm), "| with Region:", nm["Region"].notna().sum())
print("digit-count of the number after 'circ_':")
print(nm["circ_raw"].str.extract(r"circ_(\d+)")[0].str.len().value_counts(), "\n")
print(nm.head(10), "\n")

# C. what does Region look like in the unresolved rows?
reg_pat = re.compile(r"^(chr\w+):(\d+)-(\d+)$")

def region_kind(r):
    if not isinstance(r, str) or not r.strip(): return "missing"
    r = r.strip()
    if reg_pat.match(r):                        return "full chr:start-end"
    if re.match(r"^chr\w+$", r):                return "chromosome only"
    return "other format"

un = hum[~hum["in_fasta"]].copy()
un["region_kind"] = un["Region"].map(region_kind)
print("unresolved rows:", len(un))
print(un["region_kind"].value_counts(), "\n")
print(pd.crosstab(un["name_pattern"], un["region_kind"], margins=True), "\n")
print("examples of 'other format':")
print(un.loc[un["region_kind"] == "other format", "Region"].drop_duplicates().head(8).tolist())
print("\nunresolved rows that have a Gene Symbol:", un["Gene Symbol"].notna().sum())

IDs not in FASTA:
              circ_raw       circbase_id                     Region Strand Gene Symbol
547   hsa_circ_0141539  hsa_circ_0141539  chr1:2072008388-207201024      -    C1orf116
1099  hsa_circ_0141940  hsa_circ_0141940   chr3:123332641-123332832      -        MYLK
1739  hsa_circ_0142527  hsa_circ_0142527     chr8:68934270-68950527      +       PREX2
2193  hsa_circ_0141206  hsa_circ_0141206    chr14:97312431-97327072      +        VRK1
3616  hsa_circ_0141940  hsa_circ_0141940   chr3:123332641-123332832      -        MYLK 

near-miss rows: 65 | with Region: 21
digit-count of the number after 'circ_':
0
6    58
5     5
8     1
4     1
Name: count, dtype: int64 

              circ_raw                    Synonyms Region Strand Gene Symbol
106    hsa_circ_402458                         NaN   chr2      +      CCNYL1
107    hsa_circ_406329                         NaN   chr3      +       FSTL1
108    hsa_circ_407176                         NaN   chr9      +      ZCCHC7
109    hsa

## Calibrate coordinate matching on rows where we already know the answer

In [10]:
def parse_region(r):
    m = reg_pat.match(r.strip()) if isinstance(r, str) else None
    return (m.group(1), int(m.group(2)), int(m.group(3))) if m else (None, None, None)

cal = res.copy()
parsed = cal["Region"].map(parse_region)
cal["chrom_e"] = parsed.map(lambda t: t[0])
cal["start_e"] = pd.to_numeric(parsed.map(lambda t: t[1]))
cal["end_e"]   = pd.to_numeric(parsed.map(lambda t: t[2]))
cal = cal.dropna(subset=["chrom_e"]).drop_duplicates("circbase_id")
cal = cal.merge(fa_index[["circbase_id", "chrom", "start", "end", "strand"]]
                .rename(columns={"start": "start_f", "end": "end_f", "strand": "strand_f"}),
                on="circbase_id")
cal["start_f"] = pd.to_numeric(cal["start_f"])
cal["end_f"]   = pd.to_numeric(cal["end_f"])
cal["d_start"] = cal["start_e"] - cal["start_f"]
cal["d_end"]   = cal["end_e"] - cal["end_f"]

print("calibration set (resolved circRNAs with a full Region):", len(cal))
print("chromosome agrees:", (cal["chrom_e"] == cal["chrom"]).mean().round(3))
print("\nstart difference (entry - FASTA):")
print(cal["d_start"].value_counts().head(6))
print("\nend difference (entry - FASTA):")
print(cal["d_end"].value_counts().head(6))
same = (cal["d_start"].abs() <= 1) & (cal["d_end"].abs() <= 1) & (cal["chrom_e"] == cal["chrom"])
print("\nwithin +/-1 nt on both ends:", same.mean().round(3))

st = cal["Strand"].notna()
print("strand agrees (where given):", (cal.loc[st, "Strand"] == cal.loc[st, "strand_f"]).mean().round(3))
print("\nexamples of disagreement:")
print(cal.loc[~same, ["circbase_id", "Region", "chrom", "start_f", "end_f"]].head(8))

calibration set (resolved circRNAs with a full Region): 1564
chromosome agrees: 0.998

start difference (entry - FASTA):
d_start
 0.0          1539
 1.0             8
-15172.0         1
-8268529.0       1
-393778.0        1
-856510.0        1
Name: count, dtype: Int64

end difference (entry - FASTA):
d_end
 0.0          1547
 8400.0          1
-8275486.0       1
-393778.0        1
-856510.0        1
-288928.0        1
Name: count, dtype: Int64

within +/-1 nt on both ends: 0.989
strand agrees (where given): 0.996

examples of disagreement:
          circbase_id                    Region  chrom    start_f      end_f
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  196711004  196715129
379  hsa_circ_0018955    chr3:71493425-71501988  chr10   79761954   79777474
380  hsa_circ_0027821   chr12:97492460-97530859  chr12   97886238   97924637
381  hsa_circ_0005280  chr2:168012633-168075231   chr2  168869143  168931741
382  hsa_circ_0000231   chr10:31908171-31910563  chr10   32197099   3

## Tier 2, coordinate matching with verification

In [11]:
import numpy as np

# ---- parser that tolerates en-dashes, case and spaces -------------------
VALID_CHR = {str(i) for i in range(1, 23)} | {"X", "Y", "M"}

def parse_region2(r):
    if not isinstance(r, str):
        return (None, None, None)
    s = r.strip().replace("\u2013", "-").replace("\u2014", "-").replace(" ", "")
    m = re.match(r"^chr([0-9XYMxym]+):(\d+)-(\d+)$", s, re.I)
    if not m:
        return (None, None, None)
    tok = m.group(1).upper()
    start, end = int(m.group(2)), int(m.group(3))
    if tok not in VALID_CHR or start >= end:
        return (None, None, None)
    return ("chr" + tok, start, end)

# ---- FASTA lookup structures --------------------------------------------
fa = fa_index.copy()
fa["start"] = pd.to_numeric(fa["start"]).astype(float)
fa["end"]   = pd.to_numeric(fa["end"]).astype(float)
fa_by_chr   = {c: g for c, g in fa.groupby("chrom")}
fa_gene     = fa.set_index("circbase_id")["gene"].to_dict()

def match_coords(chrom, s, e, strand, tol=1):
    g = fa_by_chr.get(chrom)
    if g is None:
        return []
    hit = g[((g["start"] - s).abs() <= tol) & ((g["end"] - e).abs() <= tol)]
    if strand in ("+", "-"):
        hit = hit[hit["strand"] == strand]
    return hit["circbase_id"].tolist()

# ---- run on every row not resolved by tier 1 -----------------------------
cand = hum[~hum["in_fasta"]].copy()
parsed = cand["Region"].map(parse_region2)
cand["chrom_e"] = parsed.map(lambda t: t[0])
cand["start_e"] = parsed.map(lambda t: t[1])
cand["end_e"]   = parsed.map(lambda t: t[2])

cand_ok = cand.dropna(subset=["chrom_e"]).copy()
cand_ok["hits"] = [match_coords(c, s, e, st)
                   for c, s, e, st in zip(cand_ok["chrom_e"], cand_ok["start_e"],
                                          cand_ok["end_e"], cand_ok["Strand"])]
cand_ok["n_hits"] = cand_ok["hits"].map(len)

print("unresolved rows:", len(cand))
print("rows with a parseable full Region:", len(cand_ok))
print("\nnumber of FASTA matches per row:")
print(cand_ok["n_hits"].value_counts().sort_index())

uniq = cand_ok[cand_ok["n_hits"] == 1].copy()
uniq["matched_id"] = uniq["hits"].str[0]

# ---- verification with the gene symbol -----------------------------------
def gene_check(row):
    g_e = row["Gene Symbol"]
    g_f = fa_gene.get(row["matched_id"])
    if not isinstance(g_e, str) or g_f in (None, "None"):
        return "n/a"
    return "agree" if g_e.strip().casefold() == str(g_f).strip().casefold() else "disagree"

uniq["gene_check"] = uniq.apply(gene_check, axis=1)
print("\nunique matches:", len(uniq))
print(uniq["gene_check"].value_counts())
print("\nexamples of gene disagreement:")
print(uniq.loc[uniq["gene_check"] == "disagree",
               ["circ_raw", "Region", "Gene Symbol", "matched_id"]].head(6))

# accept unique matches that do not contradict the gene
accept = uniq[uniq["gene_check"] != "disagree"]
hum["circbase_final"] = hum["circbase_id"].where(hum["in_fasta"])
hum["match_tier"] = hum["in_fasta"].map({True: "tier1_id", False: None})
hum.loc[accept.index, "circbase_final"] = accept["matched_id"]
hum.loc[accept.index, "match_tier"] = "tier2_coords"

# ---- updated totals -------------------------------------------------------
res2   = hum[hum["circbase_final"].notna()]
pairs2 = res2[["circbase_final", "disease_std"]].drop_duplicates()
print("\nmatch tiers:")
print(hum["match_tier"].value_counts(dropna=False))
print("\nunique pairs:", len(pairs2),
      "| circRNAs:", pairs2["circbase_final"].nunique(),
      "| diseases:", pairs2["disease_std"].nunique())
print("before tier 2:  2096 pairs | 1579 circRNAs | 200 diseases")

# rows whose Region could not be parsed at all (for a manual look)
left = cand[cand["chrom_e"].isna() & cand["Region"].notna()
            & cand["Region"].astype(str).str.contains(r"\d{5,}")]
print("\nunparseable but number-like regions:", left["Region"].drop_duplicates().tolist()[:10])

unresolved rows: 1442
rows with a parseable full Region: 228

number of FASTA matches per row:
n_hits
0     86
1    142
Name: count, dtype: int64

unique matches: 142
gene_check
agree       118
n/a          18
disagree      6
Name: count, dtype: int64

examples of gene disagreement:
                circ_raw                    Region  Gene Symbol        matched_id
572          circSMARC1A  chr4:144464662-144465125      SMARC1A  hsa_circ_0001445
855   hsa_circRNA_002086    chr7:30590251-30601744    LOC401320  hsa_circ_0001693
1640           circ_2137   chr13:64564228-64608670     AK057471  hsa_circ_0030388
2049     hsa_circ_003785   chr18:32902351-32907027  XLOC_012658  hsa_circ_0003785
3158            circPVT1  chr8:128902834-128903244         PVT1  hsa_circ_0001821
3825           circ-Rtn4    chr3:17059499-17059748         RTN4  hsa_circ_0001275

match tiers:
match_tier
tier1_id        2412
NaN             1306
tier2_coords     136
Name: count, dtype: int64

unique pairs: 2208 | circRN

## Triage the tier-1 conflicts

In [12]:
cf = cal.loc[~same].copy()
same_len = ((cf["end_e"] - cf["start_e"]) - (cf["end_f"] - cf["start_f"])).abs() <= 1
chrom_ok = cf["chrom_e"] == cf["chrom"]

cf["kind"] = np.select(
    [~chrom_ok, same_len & chrom_ok],
    ["chromosome mismatch", "same length, shifted (likely other genome build)"],
    default="different length/coordinates")

print(cf["kind"].value_counts(), "\n")
print(cf[["circbase_id", "Region", "chrom", "start_f", "end_f", "kind"]].to_string())

kind
same length, shifted (likely other genome build)    8
different length/coordinates                        6
chromosome mismatch                                 3
Name: count, dtype: int64 

          circbase_id                    Region  chrom    start_f      end_f                                              kind
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  196711004  196715129                      different length/coordinates
379  hsa_circ_0018955    chr3:71493425-71501988  chr10   79761954   79777474                               chromosome mismatch
380  hsa_circ_0027821   chr12:97492460-97530859  chr12   97886238   97924637  same length, shifted (likely other genome build)
381  hsa_circ_0005280  chr2:168012633-168075231   chr2  168869143  168931741  same length, shifted (likely other genome build)
382  hsa_circ_0000231   chr10:31908171-31910563  chr10   32197099   32199491  same length, shifted (likely other genome build)
383  hsa_circ_0007930  chr5:136147831-13615

## Corrected triage, the six disagreements, and what's left

In [13]:
# ---- A. the six gene disagreements: what does the FASTA say? ------------
fa_tx = fa_index.set_index("circbase_id")["transcript"].to_dict()
dis6 = uniq[uniq["gene_check"] == "disagree"].copy()
dis6["fasta_gene"] = dis6["matched_id"].map(fa_gene)
dis6["fasta_tx"]   = dis6["matched_id"].map(fa_tx)
print("A. gene disagreements")
print(dis6[["circ_raw", "Gene Symbol", "fasta_gene", "fasta_tx", "matched_id"]].to_string(), "\n")

# ---- B. conflicts re-triaged with reversed coordinates swapped ----------
cf2 = cal.loc[~same].copy()
cf2["reversed_in_entry"] = cf2["start_e"] > cf2["end_e"]
cf2["len_e"] = (cf2["end_e"] - cf2["start_e"]).abs()
cf2["len_f"] = cf2["end_f"] - cf2["start_f"]
cf2["same_len"] = (cf2["len_e"] - cf2["len_f"]).abs() <= 1
cf2["kind"] = np.select(
    [cf2["chrom_e"] != cf2["chrom"], cf2["same_len"]],
    ["chromosome mismatch", "same length, shifted"],
    default="different length")
print("B. corrected conflict triage")
print(cf2["kind"].value_counts(), "\n")
print(cf2.loc[cf2["kind"] != "same length, shifted",
              ["circbase_id", "Region", "chrom", "start_f", "end_f", "len_e", "len_f", "kind"]].to_string(), "\n")

# ---- C. lenient re-parse of the still-unresolved rows --------------------
def parse_region3(r):
    if not isinstance(r, str):
        return (None, None, None, False)
    s = r.strip().replace("\u2013", "-").replace("\u2014", "-").replace(" ", "").rstrip(")")
    m = re.match(r"^chr([0-9XYMxym]+):(\d+)-(\d+)$", s, re.I)
    if not m:
        return (None, None, None, False)
    tok, a, b = m.group(1).upper(), int(m.group(2)), int(m.group(3))
    if tok not in VALID_CHR or a == b:
        return (None, None, None, False)
    return ("chr" + tok, min(a, b), max(a, b), a > b)

rest = hum[hum["circbase_final"].isna()].copy()
p3 = rest["Region"].map(parse_region3)
rest["chrom_e"] = p3.map(lambda t: t[0])
rest["s"]       = p3.map(lambda t: t[1])
rest["e"]       = p3.map(lambda t: t[2])
rest["rev"]     = p3.map(lambda t: t[3])
rest = rest.dropna(subset=["chrom_e"])
rest = rest[(rest["e"] - rest["s"]) <= 1_000_000]          # drops absurd spans

rest["hits"] = [match_coords(c, s, e, st)
                for c, s, e, st in zip(rest["chrom_e"], rest["s"], rest["e"], rest["Strand"])]
rest["n_hits"] = rest["hits"].map(len)
print("C. lenient parse of unresolved rows:", len(rest), "| reversed:", int(rest["rev"].sum()))
print(rest["n_hits"].value_counts().sort_index(), "\n")

def length_gene_hits(chrom, s, e, gene, tol=1):
    g = fa_by_chr.get(chrom)
    if g is None or not isinstance(gene, str):
        return []
    hit = g[((g["end"] - g["start"]) - (e - s)).abs() <= tol]
    return hit.loc[hit["gene"].astype(str).str.casefold() == gene.strip().casefold(),
                   "circbase_id"].tolist()

zero = rest[rest["n_hits"] == 0].copy()
zero["lg_hits"] = [length_gene_hits(c, s, e, g)
                   for c, s, e, g in zip(zero["chrom_e"], zero["s"], zero["e"], zero["Gene Symbol"])]
zero["n_lg"] = zero["lg_hits"].map(len)
print("zero exact hits:", len(zero), "| has a Gene Symbol:", zero["Gene Symbol"].notna().sum())
print("same chromosome + same length + same gene (possible other-build entry):")
print(zero["n_lg"].value_counts().sort_index(), "\n")
print(zero.loc[zero["n_lg"] >= 1, ["circ_raw", "Region", "Gene Symbol", "lg_hits"]].head(8).to_string())

print("\nstill unresolved overall:", hum["circbase_final"].isna().sum(), "/", len(hum))

A. gene disagreements
                circ_raw  Gene Symbol         fasta_gene           fasta_tx        matched_id
572          circSMARC1A      SMARC1A            SMARCA5          NM_003601  hsa_circ_0001445
855   hsa_circRNA_002086    LOC401320     TCONS_00012989     TCONS_00012989  hsa_circ_0001693
1640           circ_2137     AK057471  TCONS_l2_00007569  TCONS_l2_00007569  hsa_circ_0030388
2049     hsa_circ_003785  XLOC_012658     TCONS_00026319     TCONS_00026319  hsa_circ_0003785
3158            circPVT1         PVT1     TCONS_00015354     TCONS_00015354  hsa_circ_0001821
3825           circ-Rtn4         RTN4              PLCL2       NM_001144382  hsa_circ_0001275 

B. corrected conflict triage
kind
same length, shifted    11
different length         3
chromosome mismatch      3
Name: count, dtype: int64 

          circbase_id                    Region  chrom    start_f      end_f     len_e  len_f                 kind
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  1967

##  Apply the decisions, flag everything, and save

In [14]:
PLACEHOLDER = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$)", re.I)
MANUAL_OK = {"hsa_circ_0001445"}   # entry says SMARC1A, a typo for SMARCA5 (FASTA agrees on SMARCA5)

def gene_status(g_entry, g_fa, cid):
    if cid in MANUAL_OK:
        return "manual_ok"
    if not isinstance(g_entry, str) or not isinstance(g_fa, str):
        return "n/a"
    if PLACEHOLDER.match(g_entry.strip()) or PLACEHOLDER.match(g_fa.strip()):
        return "n/a"
    return "agree" if g_entry.strip().casefold() == g_fa.strip().casefold() else "disagree"

# ---- tier-2 candidates: union of the strict and the lenient parse ---------
a  = uniq[["Gene Symbol", "matched_id"]]
r1 = rest[rest["n_hits"] == 1]
b  = r1[["Gene Symbol"]].assign(matched_id=r1["hits"].str[0])
t2 = pd.concat([a, b])
t2 = t2[~t2.index.duplicated(keep="first")].copy()
t2["gene_status"] = [gene_status(g, fa_gene.get(i), i)
                     for g, i in zip(t2["Gene Symbol"], t2["matched_id"])]
print("tier-2 candidates:", len(t2))
print(t2["gene_status"].value_counts(), "\n")
t2_keep = t2[t2["gene_status"] != "disagree"]

# ---- assemble the final table ---------------------------------------------
final = hum.copy()
final["circbase_final"] = final["circbase_id"].where(final["in_fasta"])
final["match_tier"] = final["in_fasta"].map({True: "tier1_id", False: None})
final.loc[t2_keep.index, "circbase_final"] = t2_keep["matched_id"]
final.loc[t2_keep.index, "match_tier"] = "tier2_coords"

final["flag"] = final["circbase_final"].notna().map({True: "ok", False: "unresolved"})
na_idx = t2_keep.index[(t2_keep["gene_status"] == "n/a").to_numpy()]
final.loc[na_idx, "flag"] = "gene_unverifiable"
gc_idx = t2.index[(t2["gene_status"] == "disagree").to_numpy()]
final.loc[gc_idx, "flag"] = "gene_conflict"

shifted_ids  = set(cf2.loc[cf2["kind"] == "same length, shifted", "circbase_id"])
conflict_ids = set(cf2.loc[cf2["kind"] != "same length, shifted", "circbase_id"])
final.loc[final["circbase_final"].isin(shifted_ids),  "flag"] = "shifted_build"
final.loc[final["circbase_final"].isin(conflict_ids), "flag"] = "id_conflict"
final["keep"] = final["circbase_final"].notna() & ~final["circbase_final"].isin(conflict_ids)

# ---- report and save --------------------------------------------------------
print(final["flag"].value_counts(), "\n")
core  = final[final["keep"]]
pairs = core[["circbase_final", "disease_std"]].drop_duplicates()
print("KEEP rows:", len(core), "| pairs:", len(pairs),
      "| circRNAs:", pairs["circbase_final"].nunique(),
      "| diseases:", pairs["disease_std"].nunique())
print("tier-1 only was: 2096 pairs | 1579 circRNAs | 200 diseases")

cols = ["CRD ID", "circ_raw", "circbase_final", "disease_std", "match_tier",
        "flag", "keep", "PubMed ID", "Region", "Strand", "Gene Symbol"]
final[cols].to_csv(PROCESSED_DIR / "cda_entries_resolved_v1.csv", index=False)
print("saved:", PROCESSED_DIR / "cda_entries_resolved_v1.csv")

tier-2 candidates: 143
gene_status
agree        115
n/a           23
manual_ok      3
disagree       2
Name: count, dtype: int64 

flag
ok                   2506
unresolved           1299
gene_unverifiable      23
shifted_build          18
id_conflict             6
gene_conflict           2
Name: count, dtype: int64 

KEEP rows: 2547 | pairs: 2206 | circRNAs: 1644 | diseases: 209
tier-1 only was: 2096 pairs | 1579 circRNAs | 200 diseases
saved: d:\Queens_university\Project\Working\Notebook\circmtl\data\processed\cda_entries_resolved_v1.csv


## How trustworthy is the length + gene rule, and does zero-padding work?

In [15]:
# ---- 1. calibrate the length + gene fallback ------------------------------
fa2 = fa.copy()
fa2["len"]    = fa2["end"] - fa2["start"]
fa2["gene_l"] = fa2["gene"].astype(str).str.casefold()
fa_groups = {c: g for c, g in fa2.groupby("chrom")}

def lg_candidates(chrom, length, gene, tol=1):
    g = fa_groups.get(chrom)
    if g is None or not isinstance(gene, str):
        return []
    hit = g[((g["len"] - length).abs() <= tol) & (g["gene_l"] == gene.strip().casefold())]
    return hit["circbase_id"].tolist()

test = cal.dropna(subset=["Gene Symbol"]).copy()
test = test[~test["Gene Symbol"].map(lambda g: bool(PLACEHOLDER.match(str(g).strip())))]
test["len_e"] = (test["end_e"] - test["start_e"]).abs()
test["cands"] = [lg_candidates(c, l, g) for c, l, g in
                 zip(test["chrom_e"], test["len_e"], test["Gene Symbol"])]
test["n"] = test["cands"].map(len)
test["true_in"] = [t in c for t, c in zip(test["circbase_id"], test["cands"])]

print("calibration rows (resolved, real gene symbol):", len(test))
print("no candidate:", round((test["n"] == 0).mean(), 3))
u = test[test["n"] == 1]
print("exactly one candidate:", len(u), "| correct:", round(u["true_in"].mean(), 3))
m = test[test["n"] >= 2]
print("two or more candidates:", len(m), "| true ID among them:", round(m["true_in"].mean(), 3), "\n")

# ---- 2. zero-padding test on the 6-digit near-miss IDs ----------------------
nm6 = hum[near].copy()
nm6["digits"] = nm6["circ_raw"].str.extract(r"circ_(\d+)")[0]
nm6 = nm6[nm6["digits"].str.len() == 6].copy()
nm6["padded"] = "hsa_circ_0" + nm6["digits"]
nm6["padded_in_fasta"] = nm6["padded"].isin(fa_ids)

fa_c = fa.set_index("circbase_id")
def coord_agree(row):
    c, s, e, _ = parse_region3(row["Region"])
    if c is None or row["padded"] not in fa_c.index:
        return None
    r = fa_c.loc[row["padded"]]
    return bool(r["chrom"] == c and abs(r["start"] - s) <= 1 and abs(r["end"] - e) <= 1)

nm6["coord_agree"] = nm6.apply(coord_agree, axis=1)
print("6-digit near-miss rows:", len(nm6))
print("padded ID exists in FASTA:\n", nm6["padded_in_fasta"].value_counts())
print("\ncoordinates confirm the padded ID:\n", nm6["coord_agree"].value_counts(dropna=False))
print(nm6.loc[nm6["padded_in_fasta"], ["circ_raw", "padded", "Region", "coord_agree"]].head(8))

calibration rows (resolved, real gene symbol): 1534
no candidate: 0.016
exactly one candidate: 1500 | correct: 0.998
two or more candidates: 9 | true ID among them: 1.0 

6-digit near-miss rows: 58
padded ID exists in FASTA:
 padded_in_fasta
True     52
False     6
Name: count, dtype: int64

coordinates confirm the padded ID:
 coord_agree
None     43
False    10
True      5
Name: count, dtype: int64
            circ_raw            padded Region coord_agree
109  hsa_circ_082319  hsa_circ_0082319   chr7        None
210  hsa_circ_101396  hsa_circ_0101396    NaN        None
211  hsa_circ_102533  hsa_circ_0102533    NaN        None
212  hsa_circ_103783  hsa_circ_0103783    NaN        None
213  hsa_circ_102470  hsa_circ_0102470    NaN        None
732  hsa_circ_103852  hsa_circ_0103852    NaN        None
733  hsa_circ_101000  hsa_circ_0101000    NaN        None
734  hsa_circ_102086  hsa_circ_0102086    NaN        None


## Audit the flagged rows, add tier 3 as a separate column, save v2

In [16]:
# ---- A1. is "the padded ID exists in the FASTA" informative? ---------------
num = pd.to_numeric(fa_index["circbase_id"].str.extract(r"hsa_circ_(\d+)")[0],
                    errors="coerce").dropna().astype(int)
print("A1. circBase numeric ID range:", num.min(), "-", num.max(),
      "| IDs present:", len(num),
      "| density:", round(len(num) / (num.max() - num.min() + 1), 3))

# ---- A2. the gene_conflict rows (excluded) ----------------------------------
gc = t2[t2["gene_status"] == "disagree"].copy()
gc["circ_raw"]   = final.loc[gc.index, "circ_raw"]
gc["Region"]     = final.loc[gc.index, "Region"]
gc["fasta_gene"] = gc["matched_id"].map(fa_gene)
print("\nA2. gene_conflict rows:")
print(gc[["circ_raw", "Region", "Gene Symbol", "fasta_gene", "matched_id"]].to_string())

# ---- A3. rows covered by the manual SMARCA5 exception ------------------------
mo = t2[t2["gene_status"] == "manual_ok"].copy()
mo["circ_raw"] = final.loc[mo.index, "circ_raw"]
mo["Region"]   = final.loc[mo.index, "Region"]
print("\nA3. manual_ok rows:")
print(mo[["circ_raw", "Region", "Gene Symbol", "matched_id"]].to_string())

# ---- A4. near-miss rows whose coordinates could be checked -------------------
nm6["resolved_to"] = final.loc[nm6.index, "circbase_final"]
chk = nm6[nm6["coord_agree"].notna()]
print("\nA4. near-miss rows with checkable coordinates:")
print(chk[["circ_raw", "padded", "coord_agree", "resolved_to"]].to_string())

# ---- B. tier 3 as a separate column (does not touch `keep`) -------------------
t3 = zero[zero["n_lg"] == 1].copy()
t3["tier3_id"] = t3["lg_hits"].str[0]
final["tier3_id"] = t3["tier3_id"]              # aligned on index, NaN elsewhere

already = t3["tier3_id"].isin(core["circbase_final"])
print("\nB. tier-3 candidates:", len(t3))
print("candidate IDs already in the kept set:", int(already.sum()))
print("new circRNAs:", t3.loc[~already, "tier3_id"].nunique(),
      "| unique pairs:", len(t3[["tier3_id", "disease_std"]].drop_duplicates()))
print(t3[["circ_raw", "Region", "Gene Symbol", "tier3_id"]].head(8).to_string())

# ---- C. save v2 ---------------------------------------------------------------
cols2 = ["CRD ID", "circ_raw", "circbase_final", "tier3_id", "disease_std",
         "match_tier", "flag", "keep", "Expression pattern",
         "Experimental techniques", "PubMed ID", "Region", "Strand", "Gene Symbol"]
out = PROCESSED_DIR / "cda_entries_resolved_v2.csv"
final[cols2].to_csv(out, index=False)
print("\nsaved:", out)

A1. circBase numeric ID range: 1 - 140790 | IDs present: 140790 | density: 1.0

A2. gene_conflict rows:
          circ_raw                  Region Gene Symbol fasta_gene        matched_id
3825     circ-Rtn4  chr3:17059499-17059748        RTN4      PLCL2  hsa_circ_0001275
2826  circ_0021093  chr11:8714898-8728772)          ST        ST5  hsa_circ_0021093

A3. manual_ok rows:
          circ_raw                    Region Gene Symbol        matched_id
466   Circ_SMARCA5  chr4:144464661-144465125     SMARCA5  hsa_circ_0001445
572    circSMARC1A  chr4:144464662-144465125     SMARC1A  hsa_circ_0001445
1488   circSMARCA5  chr4:144464662-144465125     SMARCA5  hsa_circ_0001445

A4. near-miss rows with checkable coordinates:
             circ_raw            padded coord_agree       resolved_to
1018  hsa_circ_001242  hsa_circ_0001242       False               NaN
1183  hsa_circ_103809  hsa_circ_0103809       False               NaN
1845  hsa_circ_009755  hsa_circ_0009755       False              

## 17. What are we losing, and what do the kept sequences look like? 

In [17]:
# ---- A. who do we lose? -------------------------------------------------------
print("A1. retention by naming scheme:")
print(final.groupby("name_pattern")["keep"].agg(rows="size", kept="sum", share="mean").round(2), "\n")

d_all  = final["disease_std"].value_counts().rename("rows_all")
d_keep = final.loc[final["keep"], "disease_std"].value_counts().rename("rows_kept")
dis = pd.concat([d_all, d_keep], axis=1).fillna(0).astype(int)
dis["retention"] = (dis["rows_kept"] / dis["rows_all"]).round(2)

print("A2. ten most-reported diseases:")
print(dis.sort_values("rows_all", ascending=False).head(10), "\n")
print("diseases with zero kept rows:", int((dis["rows_kept"] == 0).sum()), "of", len(dis))
print("diseases with >= 5 kept rows:", int((dis["rows_kept"] >= 5).sum()))
print("diseases with >= 10 rows overall but retention < 0.4:")
print(dis[(dis["rows_all"] >= 10) & (dis["retention"] < 0.4)]
      .sort_values("rows_all", ascending=False), "\n")

# ---- B. sequence status of the kept circRNAs -----------------------------------
L = fa_index.set_index("circbase_id")["seq_len"]
kept = final[final["keep"]].copy()
kept["seq_len"] = kept["circbase_final"].map(L)
pp  = kept[["circbase_final", "disease_std", "seq_len"]].drop_duplicates()
ids = pp.drop_duplicates("circbase_final")

print("B1. sequence length of the kept circRNAs (nt):")
print(ids["seq_len"].describe(percentiles=[.5, .75, .9, .95]).round(0))
print("zero-length:", int((ids["seq_len"] == 0).sum()),
      "| 1-1022:", int(ids["seq_len"].between(1, 1022).sum()),
      "| 1023-5000:", int(ids["seq_len"].between(1023, 5000).sum()),
      "| > 5000:", int((ids["seq_len"] > 5000).sum()))
ok_len = pp[pp["seq_len"].between(1, 1022)]
print("\nB2. if we keep only circRNAs of 1-1022 nt:")
print("pairs:", len(ok_len), "| circRNAs:", ok_len["circbase_final"].nunique(),
      "| diseases:", ok_len["disease_std"].nunique())

# ---- C. sparsity structure -------------------------------------------------------
deg  = pp.groupby("circbase_final").size()
ddeg = pp.groupby("disease_std").size()
print("\nC1. diseases per circRNA:")
print(deg.value_counts().sort_index().head(8))
print("circRNAs with exactly one disease:", round((deg == 1).mean(), 3))
print("most-connected circRNAs:\n", deg.sort_values(ascending=False).head(5))
print("\nC2. circRNAs per disease: median", ddeg.median(),
      "| diseases with < 3 circRNAs:", int((ddeg < 3).sum()), "of", len(ddeg))

A1. retention by naming scheme:
                    rows  kept  share
name_pattern                         
arraystar_like       339    25   0.07
circbase_full       2317  2255   0.97
circbase_no_prefix   217    75   0.35
coordinate            14     3   0.21
other                967   189   0.20 

A2. ten most-reported diseases:
                            rows_all  rows_kept  retention
disease_std                                               
Gastric cancer                   277        194       0.70
Hepatocellular carcinoma         254        181       0.71
Breast cancer                    191        131       0.69
Colorectal cancer                189        138       0.73
Non-small cell lung cancer       169        115       0.68
Glioma                           133         95       0.71
Osteosarcoma                     126         77       0.61
Bladder cancer                   101         78       0.77
Ovarian cancer                    83         44       0.53
Prostate cancer    

## Confirm the cold-start share, check long circRNAs and hubs, locate the losses

In [18]:
from sklearn.model_selection import KFold

pp = (final.loc[final["keep"], ["circbase_final", "disease_std"]]
      .drop_duplicates().reset_index(drop=True))
L = fa_index.set_index("circbase_id")["seq_len"]
pp["seq_len"] = pp["circbase_final"].map(L)

# ---- A. simulate a random 5-fold edge split ---------------------------------
rows = []
for seed in (0, 1, 2):
    for k, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=seed).split(pp)):
        t = pp.iloc[te]
        rows.append({"seed": seed, "fold": k, "test_pos": len(t),
                     "cold_circ": (~t["circbase_final"].isin(set(pp.iloc[tr]["circbase_final"]))).mean(),
                     "cold_disease": (~t["disease_std"].isin(set(pp.iloc[tr]["disease_std"]))).mean()})
print("A. share of test positives with no training edge (mean over folds):")
print(pd.DataFrame(rows).groupby("seed")[["test_pos", "cold_circ", "cold_disease"]].mean().round(3), "\n")

# ---- B. length policies, long circRNAs, hubs -----------------------------------
print("B1. pairs / circRNAs / diseases by length band:")
for lo, hi in [(1, 1022), (1023, 5000), (5001, 10**9)]:
    s = pp[pp["seq_len"].between(lo, hi)]
    print(f"  {lo}-{hi}: {len(s)} pairs | {s['circbase_final'].nunique()} circRNAs "
          f"| {s['disease_std'].nunique()} diseases")
s = pp[pp["seq_len"].between(1, 5000)]
print(f"  <=5000 total: {len(s)} pairs | {s['circbase_final'].nunique()} circRNAs "
      f"| {s['disease_std'].nunique()} diseases\n")

span = fa.set_index("circbase_id")["end"] - fa.set_index("circbase_id")["start"]
lg = (pp[pp["seq_len"] > 5000].drop_duplicates("circbase_final")
      .sort_values("seq_len", ascending=False).head(8).copy())
lg["genomic_span"] = lg["circbase_final"].map(span)
lg["gene"] = lg["circbase_final"].map(fa_gene)
print("B2. longest kept circRNAs (seq_len vs genomic span):")
print(lg[["circbase_final", "seq_len", "genomic_span", "gene"]].to_string(), "\n")

deg = pp.groupby("circbase_final").size().sort_values(ascending=False)
hub = deg.head(8).rename("n_diseases").to_frame()
hub["gene"] = hub.index.map(fa_gene)
hub["seq_len"] = hub.index.map(L)
print("B3. hub circRNAs:")
print(hub.to_string(), "\n")

# ---- C. where do the losses come from? -------------------------------------------
lost = final[final["flag"] == "unresolved"]
vc = lost["PubMed ID"].value_counts()
print("C1. unresolved rows:", len(lost), "| distinct papers:", lost["PubMed ID"].nunique())
print("share from the 10 biggest papers:", round(vc.head(10).sum() / len(lost), 3))
print(vc.head(10), "\n")

dis_all = final["disease_std"].value_counts()
dis_kept = final.loc[final["keep"], "disease_std"].value_counts()
zero_dis = dis_all.index.difference(dis_kept.index)
print("C2. naming scheme of rows in the", len(zero_dis), "diseases with zero kept rows:")
print(final[final["disease_std"].isin(zero_dis)]["name_pattern"].value_counts())

A. share of test positives with no training edge (mean over folds):
      test_pos  cold_circ  cold_disease
seed                                   
0        441.2      0.640         0.036
1        441.2      0.646         0.036
2        441.2      0.645         0.035 

B1. pairs / circRNAs / diseases by length band:
  1-1022: 1665 pairs | 1232 circRNAs | 180 diseases
  1023-5000: 472 pairs | 350 circRNAs | 131 diseases
  5001-1000000000: 69 pairs | 62 circRNAs | 38 diseases
  <=5000 total: 2137 pairs | 1582 circRNAs | 207 diseases

B2. longest kept circRNAs (seq_len vs genomic span):
        circbase_final  seq_len  genomic_span       gene
2178  hsa_circ_0003420  1211004     1397938.0      ABCC1
422   hsa_circ_0033988   453671      453671.0  LINC00221
909   hsa_circ_0055538   368628      400079.0     RMND5A
865   hsa_circ_0043603   242759      246061.0      KRT19
1458  hsa_circ_0025580   161704      201335.0    SLCO1B3
861   hsa_circ_0062459    97097       97097.0       None
2036  hsa_

## Look at the auxiliary-label file before joining anything

In [19]:
cur = tables["The_manually_curated_associations_between_circRNAs_and_other_associations"][
    "CircRNA and Other Associations"]
lab_cols = ["miRNA validated", "miRNA predicted", "miRNA target", "RBP validated"]

print(cur.shape)
print("\nshare missing:\n", cur.isna().mean().round(3))
print("\nSpecies:\n", cur["Species"].value_counts(dropna=False).head(6))

for c in lab_cols:
    s = cur[c].dropna().astype(str)
    print(f"\n{c}: non-empty {len(s)}")
    print("  samples:", s.sample(min(6, len(s)), random_state=0).tolist())
    print("  separators:", {sp: int(s.str.contains(sp, regex=False).sum())
                            for sp in [",", ";", "/", "|", "\n", " and "]})

key = lambda d: (d["circRNA Name"].map(clean_txt) + "||" + d["Disease Name"].map(clean_txt)).str.casefold()
ck, ek = key(cur), key(entries)
print("\ncurated rows whose (circRNA, disease) appears in the entries table:", round(ck.isin(set(ek)).mean(), 3))
print("duplicate (circRNA, disease) keys in the curated file:", int(ck.duplicated().sum()))

(2160, 7)

share missing:
 circRNA Name       0.000
Disease Name       0.000
Species            0.000
miRNA validated    0.156
miRNA predicted    0.887
miRNA target       0.315
RBP validated      0.857
dtype: float64

Species:
 Species
Human    2027
Mouse     107
Rat        24
human       1
rats        1
Name: count, dtype: int64

miRNA validated: non-empty 1822
  samples: ['miR-421', 'miR-22-3p', 'miR-432-5p', 'miR-1179', 'miR-185-5p', 'miR-766-3p']
  separators: {',': 105, ';': 0, '/': 5, '|': 0, '\n': 0, ' and ': 5}

miRNA predicted: non-empty 244
  samples: ['hsa-miR-4778-3p, hsa-miR-5002-5p, hsa-miR-329-5p, hsa-miR-4652-3p, hsa-miR-6781-3p', 'hsa-miR-578, hsa-miR-429, hsa-miR-449a, hsa-miR-449b-5p, hsa-miR-34c-5p', 'mmu-miR-653-5p, mmu-miR-17-3p, mmu-miR-6394, mmu-miR-6948-3p, mmu-miR-148b-3p', 'hsa-miR-369-5p, hsa-miR-626, hsa-miR-326, hsa-miR-330-5p, hsa-miR-1265, hsa-miR-622', 'hsa-miR-302c-3p, hsa-miR-520d-3p, hsa-miR-302b-3p, hsa-miR-302d-3p, hsa-miR-197-5p', 'hsa-miR-138-5p,

## Name-level consistency, separators, and token counts

In [20]:
# ---- A. is name -> ID consistent? (matters before joining the curated file) ----
k = final[final["keep"]].copy()
k["name_key"] = k["circ_raw"].str.casefold()
n_ids = k.groupby("name_key")["circbase_final"].nunique()
print("A1. kept circRNA names:", len(n_ids), "| names resolving to >1 ID:", int((n_ids > 1).sum()))
bad = n_ids[n_ids > 1].index
print(k[k["name_key"].isin(bad)][["circ_raw", "circbase_final", "match_tier", "Region"]]
      .sort_values("circ_raw").head(10).to_string())

allr = final.assign(name_key=final["circ_raw"].str.casefold())
g = allr.groupby("name_key")["keep"].agg(kept="sum", rows="size")
mixed = g[(g["kept"] > 0) & (g["kept"] < g["rows"])]
print("\nA2. names kept in some rows but dropped in others:", len(mixed),
      "| dropped rows affected:", int((mixed["rows"] - mixed["kept"]).sum()))
print(mixed.head(8))

# ---- B. how are multiple values written inside a cell? ---------------------------
def show(col, pat, n=8):
    s = cur[col].dropna().astype(str)
    hit = s[s.str.contains(pat, regex=True)].drop_duplicates()
    print(f"{col} ~ {pat!r}: {len(hit)} distinct cells;", hit.head(n).tolist())

print("\nB. separators")
show("miRNA validated", r"/")
show("miRNA validated", r"\band\b")
show("miRNA validated", r",.*,")
show("RBP validated", r"[;/\n]")

# ---- C. tokens ----------------------------------------------------------------------
def tokens(s):
    if not isinstance(s, str):
        return []
    return [p.strip() for p in re.split(r"[,;\n]|\s+and\s+", s) if p.strip()]

rbp = cur["RBP validated"].map(tokens).explode().dropna()
rbp_n = rbp.str.upper().str.replace(r"\s+", "", regex=True)
print("\nC1. RBP tokens:", len(rbp_n), "| distinct:", rbp_n.nunique(),
      "| non-empty RBP cells:", int(cur["RBP validated"].notna().sum()))
print(rbp_n.value_counts().head(15).to_string())
only = cur["RBP validated"].map(lambda s: bool(tokens(s)) and
        {t.upper().replace(" ", "") for t in tokens(s)} <= {"AGO2", "IGG"})
print("rows whose RBP cell holds only AGO2 and/or IgG:", int(only.sum()))

mir = cur["miRNA validated"].map(tokens).explode().dropna()
print("\nC2. miRNA tokens:", len(mir), "| distinct as written:", mir.nunique())
print("with species prefix:", round(mir.str.match(r"(?i)^(hsa|mmu|rno)-").mean(), 3),
      "| with -3p/-5p arm:", round(mir.str.contains(r"(?i)-(?:3p|5p)$").mean(), 3))
odd = mir[~mir.str.match(r"(?i)^((hsa|mmu|rno)-)?(mir|let)-")]
print("tokens not shaped like miR-/let-:", len(odd), odd.drop_duplicates().head(12).tolist())

A1. kept circRNA names: 1791 | names resolving to >1 ID: 1
       circ_raw    circbase_final match_tier                Region
3997  circGLIS3  hsa_circ_0004221   tier1_id  chr9:3879426-3937189
4075  circGLIS3  hsa_circ_0002874   tier1_id  chr9:4286037-4286523

A2. names kept in some rows but dropped in others: 47 | dropped rows affected: 111
              kept  rows
name_key                
circ-foxm1       1     2
circ-prkci       1     3
circ-prmt5       1     3
circ-smad7       1     6
circ-znf609      1     4
circ_0000003     1     2
circ_0000502     1     2
circ_0003998     1     2

B. separators
miRNA validated ~ '/': 5 distinct cells; ['miR-127-5p/miR-612/miR-4688', 'miR-203/miR-502-5p', 'hsa-miR-337-3p/miR-337-3p', 'miR-375/1231', 'miR-375/409-3P/515-5p']
miRNA validated ~ '\\band\\b': 5 distinct cells; ['miR-339-5p, miR-96-3p and miR-135b-3p', 'miR-34, miR-34a, miR-34b, and miR-34c', 'hsa-miR-34a-5p, has-miR-221-3p, has-miR-222-3p, has-miR-181-5p, has-miR-449a, hsa-miR-125a-5p

## How much auxiliary-label coverage do the kept circRNAs have?

In [21]:
def norm_mir(t):
    t = re.sub(r"(?i)^(hsa|mmu|rno)-", "", t.strip())
    return re.sub(r"(?i)^mir-", "miR-", t)

cur_h = cur[cur["Species"].str.strip().str.lower().eq("human")].copy()
cur_h["circ_raw"] = cur_h["circRNA Name"].map(clean_txt)
cur_h["disease_std"] = cur_h["Disease Name"].map(clean_txt).str.casefold().map(dis_std)
cur_h = cur_h.dropna(subset=["disease_std"])

keyed = (final[final["keep"]][["circ_raw", "disease_std", "circbase_final"]].drop_duplicates())
m = cur_h.merge(keyed, on=["circ_raw", "disease_std"], how="inner")
print("human curated rows:", len(cur_h), "| joined to the kept set:", len(m),
      "| circRNAs:", m["circbase_final"].nunique())

mir_e = (m.assign(mirna=m["miRNA validated"].map(tokens)).explode("mirna").dropna(subset=["mirna"]))
mir_e["mirna"] = mir_e["mirna"].map(norm_mir)
mir_e = mir_e[["circbase_final", "mirna"]].drop_duplicates()

rbp_e = (m.assign(rbp=m["RBP validated"].map(tokens)).explode("rbp").dropna(subset=["rbp"]))
rbp_e["rbp"] = rbp_e["rbp"].str.upper().str.replace(r"\s+", "", regex=True)
rbp_e = rbp_e[rbp_e["rbp"] != "IGG"][["circbase_final", "rbp"]].drop_duplicates()
rbp_x = rbp_e[rbp_e["rbp"] != "AGO2"]

pp5 = pp[pp["seq_len"].between(1, 5000)]
ids = set(pp5["circbase_final"])
deg = pp5.groupby("circbase_final").size()
deg1 = set(deg[deg == 1].index)
S = lambda d: set(d["circbase_final"]) & ids

print(f"\ncircRNAs in the <=5000 nt set: {len(ids)} (degree-1: {len(deg1)})")
for name, d in [("validated miRNA", mir_e), ("any RBP (IgG removed)", rbp_e), ("RBP other than AGO2", rbp_x)]:
    s = S(d)
    print(f"{name:24s}: {len(s):5d} circRNAs ({len(s)/len(ids):.1%}) | "
          f"among degree-1: {len(s & deg1):5d} ({len(s & deg1)/len(deg1):.1%})")
print("miRNA and non-AGO2 RBP both:", len(S(mir_e) & S(rbp_x)))

mk = mir_e[mir_e["circbase_final"].isin(ids)]
share = mk.groupby("mirna")["circbase_final"].nunique()
print("\nmiRNA edges:", len(mk), "| distinct miRNAs:", len(share),
      "| miRNAs linked to >=2 circRNAs:", int((share >= 2).sum()))
rk = rbp_x[rbp_x["circbase_final"].isin(ids)]
print("non-AGO2 RBP edges:", len(rk), "| distinct RBPs:", rk["rbp"].nunique())
print(rk["rbp"].value_counts().head(10).to_string())

human curated rows: 2028 | joined to the kept set: 1321 | circRNAs: 857

circRNAs in the <=5000 nt set: 1582 (degree-1: 1267)
validated miRNA         :   697 circRNAs (44.1%) | among degree-1:   458 (36.1%)
any RBP (IgG removed)   :   179 circRNAs (11.3%) | among degree-1:   105 (8.3%)
RBP other than AGO2     :    73 circRNAs (4.6%) | among degree-1:    43 (3.4%)
miRNA and non-AGO2 RBP both: 49

miRNA edges: 1081 | distinct miRNAs: 587 | miRNAs linked to >=2 circRNAs: 222
non-AGO2 RBP edges: 124 | distinct RBPs: 94
rbp
ANTI-AGO2                5
ANTI-IGG                 5
CYP51A1                  4
FKBP5                    4
HUR                      3
EZH2                     3
AGO3                     3
ANTI-ARGONAUTE2(AGO2)    3
CDK6                     2
CYCLIND1                 2


## Name propagation within a paper, and the no-prefix hypothesis

In [22]:
final["name_key"] = final["circ_raw"].str.casefold()
core = final[final["keep"]]

# ---- A. circGLIS3: same name, two IDs ---------------------------------------
g = final[final["name_key"] == "circglis3"]
print("A. circGLIS3 rows:")
print(g[["circ_raw", "disease_std", "PubMed ID", "circbase_final", "Region", "flag"]].to_string(), "\n")

# ---- B. within-paper name propagation ----------------------------------------
lookup = (core.groupby(["name_key", "PubMed ID"])["circbase_final"]
          .agg(lambda s: sorted(set(s))).to_dict())
dropped = final[~final["keep"] & final["flag"].eq("unresolved")].copy()
dropped["cand"] = [lookup.get((n, p), []) for n, p in zip(dropped["name_key"], dropped["PubMed ID"])]
dropped["n_cand"] = dropped["cand"].map(len)
print("B. unresolved rows:", len(dropped))
print(dropped["n_cand"].value_counts().sort_index())

prop = dropped[dropped["n_cand"] == 1].copy()
prop["cand_id"] = prop["cand"].str[0]
have = set(map(tuple, core[["circbase_final", "disease_std"]].drop_duplicates().values))
new_pairs = {(i, d) for i, d in zip(prop["cand_id"], prop["disease_std"])} - have
print("within-paper candidates:", len(prop), "| pairs not already kept:", len(new_pairs))
print(prop[["circ_raw", "disease_std", "PubMed ID", "cand_id"]].head(8).to_string(), "\n")

# ---- C. the no-prefix hypothesis -----------------------------------------------
shape = r"(?i)^circ[_-]?\d{7}$"
np_k = final[final["circ_raw"].str.match(shape) & final["keep"]].copy()
np_k["implied"] = "hsa_circ_" + np_k["circ_raw"].str.extract(r"(\d{7})")[0]
print("C. no-prefix names already resolved:", len(np_k), "\n   by tier:")
print(np_k["match_tier"].value_counts())
print("resolved ID equals the implied hsa_circ_ ID:",
      round((np_k["circbase_final"] == np_k["implied"]).mean(), 3))
print(np_k[["circ_raw", "implied", "circbase_final", "Region", "match_tier"]].head(8).to_string())
print("unresolved rows with this name shape:",
      int((final["circ_raw"].str.match(shape) & ~final["keep"]).sum()))

A. circGLIS3 rows:
       circ_raw     disease_std   PubMed ID    circbase_final                Region flag
3997  circGLIS3     Lung cancer  33520363.0  hsa_circ_0004221  chr9:3879426-3937189   ok
4075  circGLIS3  Bladder cancer  33656636.0  hsa_circ_0002874  chr9:4286037-4286523   ok 

B. unresolved rows: 1299
n_cand
0    1298
1       1
Name: count, dtype: int64
within-paper candidates: 1 | pairs not already kept: 0
        circ_raw                         disease_std   PubMed ID           cand_id
3452  circ-SMAD7  Esophageal squamous cell carcinoma  30611100.0  hsa_circ_0000848 

C. no-prefix names already resolved: 63 
   by tier:
match_tier
tier1_id        44
tier2_coords    19
Name: count, dtype: int64
resolved ID equals the implied hsa_circ_ ID: 0.968
          circ_raw           implied    circbase_final                     Region    match_tier
1256  circ_0030235  hsa_circ_0030235  hsa_circ_0030235    chr13:49075877-49077050  tier2_coords
1283  circ_0008450  hsa_circ_0008450  hs

## Normalize the labels properly and recount coverage

In [23]:
HYPH = dict.fromkeys(map(ord, "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"), "-")

def split_mir(s):
    if not isinstance(s, str):
        return []
    s = s.translate(HYPH)
    return [p.strip() for p in re.split(r"[,;\n]|\s+and\s+|/", s) if p.strip()]

def norm_mirna(t):
    t = t.translate(HYPH).strip()
    t = re.sub(r"(?i)^(hsa|has|mmu|rno)[-_]", "", t)
    t = re.sub(r"(?i)^(microrna|mirna|mir)[-_ ]?", "miR-", t)
    t = re.sub(r"^(\d)", r"miR-\1", t)                 # shorthand after '/', e.g. '1231'
    t = re.sub(r"(?i)^let[-_ ]?", "let-", t)
    arm = re.search(r"(?i)-(3p|5p)$", t)
    return re.sub(r"(?i)-(3p|5p)$", "", t), (arm.group(1).lower() if arm else None)

VALID = r"^(miR-\d+[A-Za-z]*(-\d+)?|let-\d+[A-Za-z]*)$"

def norm_rbp(t):
    u = re.sub(r"\s+", "", t.translate(HYPH)).upper().strip("()")
    u = re.sub(r"^ANTI-?", "", u)
    if "IGG" in u:
        return None                                   # negative-control antibody
    return "AGO2" if ("AGO2" in u or "ARGONAUTE2" in u) else u

# ---- join-key check ------------------------------------------------------------
dupkey = keyed.duplicated(["circ_raw", "disease_std"], keep=False)
print("(name, disease) keys that map to >1 circBase ID:", int(dupkey.sum()))
print(keyed[dupkey].head(6).to_string(), "\n")

# ---- miRNA ------------------------------------------------------------------------
mir = m.assign(tok=m["miRNA validated"].map(split_mir)).explode("tok").dropna(subset=["tok"]).copy()
pr = mir["tok"].map(norm_mirna)
mir["mirna"] = pr.map(lambda t: t[0])
mir["arm"]   = pr.map(lambda t: t[1])
ok = mir["mirna"].str.match(VALID)
print("miRNA tokens:", len(mir), "| not matching the miR-/let- shape:", int((~ok).sum()))
print(mir.loc[~ok, "tok"].drop_duplicates().head(15).tolist())
mir_e = mir[ok][["circbase_final", "mirna"]].drop_duplicates()

mk = mir_e[mir_e["circbase_final"].isin(ids)]
cnt = mk.groupby("mirna")["circbase_final"].nunique()
shared = set(cnt[cnt >= 2].index)
conn = set(mk.loc[mk["mirna"].isin(shared), "circbase_final"])
cov = set(mk["circbase_final"])
print("\nmiRNA edges:", len(mk), "| distinct miRNAs (arm-stripped):", len(cnt),
      "| linked to >=2 circRNAs:", len(shared))
print("circRNAs with a miRNA edge:", len(cov), f"({len(cov)/len(ids):.1%})",
      "| degree-1:", len(cov & deg1), f"({len(cov & deg1)/len(deg1):.1%})")
print("circRNAs sharing a miRNA with another circRNA:", len(conn),
      "| degree-1:", len(conn & deg1))
print("most shared miRNAs:\n", cnt.sort_values(ascending=False).head(8).to_string())

# ---- RBP ----------------------------------------------------------------------------
rb = m.assign(tok=m["RBP validated"].map(tokens)).explode("tok").dropna(subset=["tok"]).copy()
rb["rbp"] = rb["tok"].map(norm_rbp)
rb = rb.dropna(subset=["rbp"])
rb_e = rb[["circbase_final", "rbp"]].drop_duplicates()
rb_e = rb_e[rb_e["circbase_final"].isin(ids)]
cr = rb_e.groupby("rbp")["circbase_final"].nunique().sort_values(ascending=False)
non = rb_e[rb_e["rbp"] != "AGO2"]
print("\nRBP edges (IgG removed):", len(rb_e), "| distinct:", len(cr), "| seen with >=2 circRNAs:", int((cr >= 2).sum()))
print(cr.head(15).to_string())
print("non-AGO2: edges", len(non), "| circRNAs", non["circbase_final"].nunique(),
      "| distinct RBPs", non["rbp"].nunique())
print("rbp names that are not RBP-shaped (contain digits/dashes), first 15:",
      sorted({r for r in non["rbp"] if re.search(r"[0-9\-]", r)})[:15])

(name, disease) keys that map to >1 circBase ID: 0
Empty DataFrame
Columns: [circ_raw, disease_std, circbase_final]
Index: [] 

miRNA tokens: 1226 | not matching the miR-/let- shape: 3
['miR-101-3p.1', 'miR-203a-3p.1', 'miR-NC']

miRNA edges: 1055 | distinct miRNAs (arm-stripped): 406 | linked to >=2 circRNAs: 217
circRNAs with a miRNA edge: 695 (43.9%) | degree-1: 457 (36.1%)
circRNAs sharing a miRNA with another circRNA: 593 | degree-1: 378
most shared miRNAs:
 mirna
miR-145     24
miR-149     17
miR-338     17
miR-326     17
miR-130a    13
miR-182     13
miR-370     12
miR-330     11

RBP edges (IgG removed): 230 | distinct: 83 | seen with >=2 circRNAs: 14
rbp
AGO2        128
FKBP5         4
CYP51A1       4
HUR           3
EZH2          3
AGO3          3
FUS           2
AUF1          2
CYCLIND1      2
CDK6          2
QKI           2
MMP-2         2
AGO1          2
STAT3         2
AGO           1
non-AGO2: edges 102 | circRNAs 61 | distinct RBPs 82
rbp names that are not RBP-shaped (

## Test the no-prefix rule on the unresolved rows

In [24]:
shape = r"(?i)^circ[_-]?\d{7}$"
fa_c = fa.set_index("circbase_id")

def coord_check(region, cid):
    c, s, e, _ = parse_region3(region)
    if c is None or cid not in fa_c.index:
        return None
    r = fa_c.loc[cid]
    return bool(r["chrom"] == c and abs(r["start"] - s) <= 1 and abs(r["end"] - e) <= 1)

npn = final[final["circ_raw"].str.match(shape)].copy()
npn["implied"] = "hsa_circ_" + npn["circ_raw"].str.extract(r"(\d{7})")[0]

# ---- A. the two disagreements among resolved rows ----------------------------
res_np = npn[npn["keep"]]
bad = res_np[res_np["circbase_final"] != res_np["implied"]]
print("A. resolved no-prefix rows where the ID differs from the name:")
print(bad[["circ_raw", "implied", "circbase_final", "match_tier", "Region", "Gene Symbol"]].to_string(), "\n")

# ---- B. FASTA ID density: is 'exists in FASTA' informative? --------------------
num = pd.to_numeric(fa_index["circbase_id"].str.extract(r"hsa_circ_(\d+)")[0],
                    errors="coerce").dropna().astype(int)
print("B. FASTA numeric IDs:", num.min(), "-", num.max(),
      "| density:", round(len(num) / (num.max() - num.min() + 1), 3), "\n")

# ---- C. the unresolved rows ------------------------------------------------------
un = npn[~npn["keep"]].copy()
un["in_fasta"] = un["implied"].isin(fa_ids)
un["coord"] = [coord_check(r, i) for r, i in zip(un["Region"], un["implied"])]
un["gene"] = [gene_status(g, fa_gene.get(i), i) for g, i in zip(un["Gene Symbol"], un["implied"])]
print("C. unresolved no-prefix rows:", len(un))
print("implied ID in FASTA:\n", un["in_fasta"].value_counts())
print("\ncoordinates vs implied ID:\n", un["coord"].value_counts(dropna=False))
print("\ngene vs implied ID:\n", un["gene"].value_counts())
print("\nwith a full Region:", int(un["Region"].map(lambda r: parse_region3(r)[0] is not None).sum()))
print("new circRNA-disease pairs if all in-FASTA IDs were accepted:",
      len(set(zip(un.loc[un["in_fasta"], "implied"], un.loc[un["in_fasta"], "disease_std"]))
          - set(map(tuple, core[["circbase_final", "disease_std"]].values))))

A. resolved no-prefix rows where the ID differs from the name:
          circ_raw           implied    circbase_final    match_tier                   Region Gene Symbol
1788  circ_0095812  hsa_circ_0095812  hsa_circ_0000253  tier2_coords  chr10:97999787-97999925        BLNK
3236  circ_0007841  hsa_circ_0007841  hsa_circ_0007842  tier2_coords   chr7:73654267-73661093        RFC2 

B. FASTA numeric IDs: 1 - 140790 | density: 1.0 

C. unresolved no-prefix rows: 133
implied ID in FASTA:
 in_fasta
True    133
Name: count, dtype: int64

coordinates vs implied ID:
 coord
None     131
True       1
False      1
Name: count, dtype: int64

gene vs implied ID:
 gene
n/a         119
agree        13
disagree      1
Name: count, dtype: int64

with a full Region: 2
new circRNA-disease pairs if all in-FASTA IDs were accepted: 87


## Does sharing a miRNA predict sharing a disease across papers?

In [25]:
from itertools import combinations

dis_of    = pp5.groupby("circbase_final")["disease_std"].agg(set).to_dict()
papers_of = (final[final["keep"]].groupby("circbase_final")["PubMed ID"]
             .agg(lambda s: set(s.dropna())).to_dict())

pairs = set()
for members in mk.groupby("mirna")["circbase_final"].agg(lambda s: sorted(set(s))):
    pairs.update(combinations(members, 2))

df = pd.DataFrame(list(pairs), columns=["a", "b"])
df = df[df["a"].isin(dis_of) & df["b"].isin(dis_of)].copy()
df["share_disease"] = [bool(dis_of[a] & dis_of[b]) for a, b in zip(df["a"], df["b"])]
df["share_paper"]   = [bool(papers_of.get(a, set()) & papers_of.get(b, set())) for a, b in zip(df["a"], df["b"])]

print("circRNA pairs sharing >=1 miRNA:", len(df))
print(df.groupby("share_paper")["share_disease"].agg(pairs="size", frac_share_disease="mean").round(3), "\n")

def homophily(pr, n_perm=500, seed=0):
    names = sorted(set(pr["a"]) | set(pr["b"]))
    pos = {c: i for i, c in enumerate(names)}
    sets = [dis_of[c] for c in names]
    ai, bi = pr["a"].map(pos).to_numpy(), pr["b"].map(pos).to_numpy()
    def hit(p):
        return np.mean([bool(sets[p[i]] & sets[p[j]]) for i, j in zip(ai, bi)])
    obs = hit(np.arange(len(names)))
    rng = np.random.default_rng(seed)
    null = np.array([hit(rng.permutation(len(names))) for _ in range(n_perm)])
    return obs, null.mean(), (np.sum(null >= obs) + 1) / (n_perm + 1), len(pr), len(names)

cross = df[~df["share_paper"]]
low = cross[cross["a"].map(lambda c: len(dis_of[c]) <= 2) & cross["b"].map(lambda c: len(dis_of[c]) <= 2)]
print("different-paper pairs, observed vs shuffled-disease null:")
for name, pr in [("all different-paper pairs", cross), ("both circRNAs with <=2 diseases", low)]:
    if len(pr) == 0:
        print(name, ": no pairs"); continue
    obs, nm, p, npairs, nc = homophily(pr)
    print(f"{name}: pairs={npairs}, circRNAs={nc} | share a disease: observed={obs:.3f}, null={nm:.3f}, perm p={p:.3f}")

circRNA pairs sharing >=1 miRNA: 2250
             pairs  frac_share_disease
share_paper                           
False         2208               0.189
True            42               1.000 

different-paper pairs, observed vs shuffled-disease null:
all different-paper pairs: pairs=2208, circRNAs=591 | share a disease: observed=0.189, null=0.089, perm p=0.002
both circRNAs with <=2 diseases: pairs=1163, circRNAs=469 | share a disease: observed=0.074, null=0.049, perm p=0.002


## Cell 25b: Apply the one override, then run Cells 26 and 27

In [26]:
# Row-level override. Evidence: the name implies the ID, coordinates match uniquely,
# and the entry gene 'ST' is a truncation of 'ST5' (so gene_status wrongly said 'disagree').
OVERRIDES = {
    2826: ("circ_0021093", "hsa_circ_0021093"),   # index: (expected circ_raw, ID)
}

before = set(map(tuple, final.loc[final["keep"], ["circbase_final", "disease_std"]]
                 .drop_duplicates().values))

for idx, (raw, cid) in OVERRIDES.items():
    assert final.loc[idx, "circ_raw"] == raw, f"index {idx} changed - check before patching"
    final.loc[idx, "circbase_final"] = cid
    final.loc[idx, "match_tier"] = "tier2_coords"
    final.loc[idx, "flag"] = "ok"
    final.loc[idx, "keep"] = True

core = final[final["keep"]]
after = core[["circbase_final", "disease_std"]].drop_duplicates()
print("pairs:", len(before), "->", len(after),
      "| circRNAs:", after["circbase_final"].nunique(),
      "| diseases:", after["disease_std"].nunique())
print(final.loc[list(OVERRIDES), ["circ_raw", "circbase_final", "disease_std", "flag", "keep"]])
print("gene_conflict rows left:", int((final["flag"] == "gene_conflict").sum()))

pairs: 2206 -> 2206 | circRNAs: 1644 | diseases: 209
          circ_raw    circbase_final               disease_std flag  keep
2826  circ_0021093  hsa_circ_0021093  Hepatocellular carcinoma   ok  True
gene_conflict rows left: 1


## Build the v3 table with flagged sensitivity tiers

In [27]:
shape = r"(?i)^circ[_-]?\d{7}$"
v3 = final.copy()

if "tier3_id" not in v3.columns:
    print("WARNING: Cell 16 was not run, so tier-3 candidates are omitted.")
    v3["tier3_id"] = np.nan

# ---- implied-prefix candidates (unresolved rows only) ---------------------------
mask = v3["circ_raw"].str.match(shape) & v3["flag"].eq("unresolved")
un = v3[mask].copy()
un["implied"] = "hsa_circ_" + un["circ_raw"].str.extract(r"(\d{7})")[0]
un = un[un["implied"].isin(fa_ids)]
un["coord"] = [coord_check(r, i) for r, i in zip(un["Region"], un["implied"])]
un["gene"]  = [gene_status(g, fa_gene.get(i), i) for g, i in zip(un["Gene Symbol"], un["implied"])]

conflict = ((un["coord"] == False) | (un["gene"] == "disagree")).to_numpy()
verified = (((un["coord"] == True) | (un["gene"] == "agree")).to_numpy()) & ~conflict
status = np.where(conflict, "conflict", np.where(verified, "verified", "unverified"))

v3["implied_id"] = pd.Series(un["implied"].to_numpy(), index=un.index)
v3["implied_status"] = pd.Series(status, index=un.index)
print("implied-prefix candidates:", len(un))
print(v3["implied_status"].value_counts(dropna=False), "\n")

# ---- sensitivity ID: primary ID, else implied (not conflict), else tier 3 -----------
imp_ok = v3["implied_id"].where(v3["implied_status"].isin(["verified", "unverified"]))
v3["circbase_sens"] = v3["circbase_final"].where(v3["keep"]).combine_first(imp_ok).combine_first(v3["tier3_id"])
v3["keep_sensitivity"] = v3["circbase_sens"].notna()

def summarize(label, idcol, mask):
    d = v3[mask][[idcol, "disease_std"]].drop_duplicates()
    print(f"{label:34s} pairs={len(d):5d} | circRNAs={d[idcol].nunique():5d} | diseases={d['disease_std'].nunique():4d}")

summarize("primary (tiers 1+2)", "circbase_final", v3["keep"])
summarize("sensitivity (+ implied + tier 3)", "circbase_sens", v3["keep_sensitivity"])

cols = ["CRD ID", "circ_raw", "circbase_final", "implied_id", "implied_status", "tier3_id",
        "circbase_sens", "disease_std", "match_tier", "flag", "keep", "keep_sensitivity",
        "Expression pattern", "Experimental techniques", "PubMed ID", "Region", "Strand", "Gene Symbol"]
v3[cols].to_csv(PROCESSED_DIR / "cda_entries_resolved_v3.csv", index=False)
print("saved:", PROCESSED_DIR / "cda_entries_resolved_v3.csv")

implied-prefix candidates: 132
implied_status
NaN           3722
unverified     119
verified        12
conflict         1
Name: count, dtype: int64 

primary (tiers 1+2)                pairs= 2206 | circRNAs= 1644 | diseases= 209
sensitivity (+ implied + tier 3)   pairs= 2319 | circRNAs= 1715 | diseases= 215
saved: d:\Queens_university\Project\Working\Notebook\circmtl\data\processed\cda_entries_resolved_v3.csv


## Degree-controlled version of the miRNA test

In [28]:
def build_pairs(max_circ=None):
    cnt = mk.groupby("mirna")["circbase_final"].nunique()
    keep_m = set(cnt.index) if max_circ is None else set(cnt[cnt <= max_circ].index)
    prs = set()
    grp = mk[mk["mirna"].isin(keep_m)].groupby("mirna")["circbase_final"].agg(lambda s: sorted(set(s)))
    for members in grp:
        prs.update(combinations(members, 2))
    d = pd.DataFrame(list(prs), columns=["a", "b"])
    d = d[d["a"].isin(dis_of) & d["b"].isin(dis_of)].copy()
    d["share_paper"] = [bool(papers_of.get(a, set()) & papers_of.get(b, set())) for a, b in zip(d["a"], d["b"])]
    return d[~d["share_paper"]]

def strat_homophily(pr, n_perm=1000, seed=0):
    names = sorted(set(pr["a"]) | set(pr["b"]))
    pos = {c: i for i, c in enumerate(names)}
    sets = [dis_of[c] for c in names]
    bins = np.array([min(len(s), 3) for s in sets])
    ai, bi = pr["a"].map(pos).to_numpy(), pr["b"].map(pos).to_numpy()
    idx_by_bin = {b: np.where(bins == b)[0] for b in np.unique(bins)}

    def hit(order):
        return np.mean([bool(sets[order[i]] & sets[order[j]]) for i, j in zip(ai, bi)])

    obs = hit(np.arange(len(names)))
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        order = np.arange(len(names))
        for idx in idx_by_bin.values():
            order[idx] = rng.permutation(idx)
        null.append(hit(order))
    null = np.array(null)
    return obs, null.mean(), (np.sum(null >= obs) + 1) / (n_perm + 1), len(pr)

rows = []
for mlabel, mx in [("all miRNAs", None), ("miRNAs on <=10 circRNAs", 10)]:
    pr_all = build_pairs(mx)
    nd = lambda c: len(dis_of[c])
    subsets = [("all different-paper pairs", pr_all),
               ("both circRNAs <=2 diseases", pr_all[pr_all["a"].map(nd).le(2) & pr_all["b"].map(nd).le(2)]),
               ("both circRNAs exactly 1 disease", pr_all[pr_all["a"].map(nd).eq(1) & pr_all["b"].map(nd).eq(1)])]
    for slabel, pr in subsets:
        if len(pr) < 50:
            rows.append((mlabel, slabel, len(pr), np.nan, np.nan, np.nan, np.nan)); continue
        obs, nm, p, n = strat_homophily(pr)
        rows.append((mlabel, slabel, n, round(obs, 3), round(nm, 3), round(obs / nm, 2), round(p, 3)))

out = pd.DataFrame(rows, columns=["miRNA set", "pair set", "pairs", "observed", "null (degree-matched)", "lift", "perm p"])
print(out.to_string(index=False))
print("\n(p-value floor with 1000 permutations = 0.001)")

              miRNA set                        pair set  pairs  observed  null (degree-matched)  lift  perm p
             all miRNAs       all different-paper pairs   2208     0.189                  0.136  1.39   0.001
             all miRNAs      both circRNAs <=2 diseases   1163     0.074                  0.054  1.38   0.005
             all miRNAs both circRNAs exactly 1 disease    554     0.043                  0.033  1.30   0.116
miRNAs on <=10 circRNAs       all different-paper pairs   1285     0.184                  0.143  1.29   0.001
miRNAs on <=10 circRNAs      both circRNAs <=2 diseases    648     0.076                  0.054  1.40   0.018
miRNAs on <=10 circRNAs both circRNAs exactly 1 disease    304     0.053                  0.035  1.50   0.073

(p-value floor with 1000 permutations = 0.001)


## Save the frozen tables

In [29]:
L = fa_index.set_index("circbase_id")["seq_len"]

# ---- primary pair table -----------------------------------------------------------------
prim = (final.loc[final["keep"], ["circbase_final", "disease_std"]]
        .drop_duplicates().rename(columns={"circbase_final": "circrna_id", "disease_std": "disease"}))
prim["seq_len"] = prim["circrna_id"].map(L)
prim.to_csv(PROCESSED_DIR / "cda_pairs_primary.csv", index=False)

# ---- circRNA table: length policy, degree, miRNA coverage --------------------------------
mir_ok = mir[mir["mirna"].str.match(VALID)]
mir_edges = (mir_ok[["circbase_final", "mirna", "arm"]].drop_duplicates()
             .rename(columns={"circbase_final": "circrna_id"}))
mir_edges.to_csv(PROCESSED_DIR / "mirna_edges_curated.csv", index=False)

circ = prim.groupby("circrna_id").agg(n_diseases=("disease", "size"), seq_len=("seq_len", "first")).reset_index()
circ["length_band"] = np.select([circ["seq_len"].between(1, 1022), circ["seq_len"].between(1023, 5000)],
                                ["<=1022", "1023-5000"], default=">5000")
circ["use_primary"] = circ["length_band"] != ">5000"
circ["has_mirna_edge"] = circ["circrna_id"].isin(set(mir_edges["circrna_id"]))
circ.to_csv(PROCESSED_DIR / "circrna_table.csv", index=False)

# ---- sanity prints ------------------------------------------------------------------------
print("pairs:", len(prim), "| circRNAs:", len(circ), "| diseases:", prim["disease"].nunique())
print(circ["length_band"].value_counts().to_string())
print("model-input circRNAs (<=5000 nt):", int(circ["use_primary"].sum()))
print("miRNA edges:", len(mir_edges), "| circRNAs with a miRNA edge:", int(circ["has_mirna_edge"].sum()))
print("saved to", PROCESSED_DIR)

pairs: 2206 | circRNAs: 1644 | diseases: 209
length_band
<=1022       1232
1023-5000     350
>5000          62
model-input circRNAs (<=5000 nt): 1582
miRNA edges: 1105 | circRNAs with a miRNA edge: 722
saved to d:\Queens_university\Project\Working\Notebook\circmtl\data\processed


## Probe the API with two small queries

In [30]:
import io, urllib.request, urllib.error

ENCORI = "https://rnasysu.com/encori/api"

def probe(endpoint, params, n=6, timeout=180):
    qs = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{ENCORI}/{endpoint}/?{qs}"
    print("URL:", url)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            status, txt = r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.URLError as e:
        print("request failed:", e)
        return None
    lines = [l for l in txt.splitlines() if l.strip()]
    meta = [l for l in lines if l.startswith("#")]
    data = [l for l in lines if not l.startswith("#")]
    print("HTTP", status, "| comment lines:", len(meta), "| data lines incl. header:", len(data))
    for l in meta[:3]:
        print("  #", l[:160])
    for l in data[:n]:
        print("  ", l[:260])
    print()
    return data

def to_df(data):
    if not data or len(data) < 2 or "\t" not in data[0]:
        return None
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

# rbp_lines = probe("RBPTarget", {"assembly": "hg19", "geneType": "circRNA", "RBP": "QKI",
#                                 "clipExpNum": 1, "pancancerNum": 0,
#                                 "target": "all", "cellType": "all"})

# mir_lines = probe("miRNATarget", {"assembly": "hg19", "geneType": "circRNA",
#                                   "miRNA": "hsa-miR-145-5p", "clipExpNum": 1,
#                                   "degraExpNum": 0, "pancancerNum": 0,
#                                   "programNum": 1, "program": "None",
#                                   "target": "all", "cellType": "all"})


# 1. Is the whole API down, or just circRNA?
rbp_lines = probe("RBPTarget", {"assembly": "hg19", "geneType": "mRNA", "RBP": "IGF2BP1",
                    "clipExpNum": 5, "pancancerNum": 0, "target": "all", "cellType": "all"})

# 2. Does a different assembly work?
mir_lines = probe("RBPTarget", {"assembly": "hg38", "geneType": "circRNA", "RBP": "QKI",
                    "clipExpNum": 1, "pancancerNum": 0, "target": "all", "cellType": "all"})

for name, lines in [("RBPTarget (QKI)", rbp_lines), ("miRNATarget (miR-145-5p)", mir_lines)]:
    d = to_df(lines)
    print("=" * 80, "\n", name)
    if d is None:
        print("not a tab-separated table - see the raw lines above")
        continue
    print("shape:", d.shape)
    print("columns:", d.columns.tolist())
    print(d.head(3).T.to_string())

URL: https://rnasysu.com/encori/api/RBPTarget/?assembly=hg19&geneType=mRNA&RBP=IGF2BP1&clipExpNum=5&pancancerNum=0&target=all&cellType=all
HTTP 200 | comment lines: 0 | data lines incl. header: 3
   <br />
   <b>Warning</b>:  mysqli_error() expects exactly 1 parameter, 0 given in <b>/www/wwwroot/html/encori/api/baseFunc.php</b> on line <b>20</b><br />
   Fail to qurey!

URL: https://rnasysu.com/encori/api/RBPTarget/?assembly=hg38&geneType=circRNA&RBP=QKI&clipExpNum=1&pancancerNum=0&target=all&cellType=all
HTTP 200 | comment lines: 2 | data lines incl. header: 5572
  # #please cite:
  # #Keren Zhou, Junhong Huang, Shurong Liu, Wujian Zheng, Shun Liu, Yinghao Xing, Yinan Cao, Yibo Chen, Xinping Hu, Li Cai, Jie Zhou, Penghui Lin, Jiajia Xuan, Fan
   RBP	geneID	geneName	txIDcat	geneType	clusterNum	totalClipExpNum	totalClipSiteNum	clusterID	chromosome	narrowStart	narrowEnd	broadStart	broadEnd	strand	clipExpNum	cellline/tissue
   QKI	NM_000028	AGL	hsa_circ_0013297	circRNA	1	1	1	QKI:CH435	chr